# 17a Latency or channel mixing? (THINGS-EEG1 development people)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 08a and 08b (only the 22 development people are opened).

Development analysis, not a prespecified test. Is the personal part of the map a time shift rather than channel mixing? For each person and budget (k = 100,
250, 500), borrowing is shifted by the lag (-30 to +30 ms) that best matches the person's labelled images at 0 to
400 ms; scored: the shift alone, the map alone, and the map fitted after the shift.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '17a-latency', 'cpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"ks": [100, 250, 500], "draws": [13, 14, 15], "shifts": [-3, -2, -1, 0, 1, 2, 3], "match_window_s": [0.0, 0.4], "window_s": [0.1, 0.2], "map_window_ms": 100, "dataset": "EEG1"}')
TAG, DATASET = '17a', 'EEG1'
PEOPLE = ['sub-04', 'sub-07', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15', 'sub-17', 'sub-20', 'sub-21', 'sub-24', 'sub-25', 'sub-26', 'sub-29', 'sub-30', 'sub-34', 'sub-35', 'sub-38', 'sub-39', 'sub-46', 'sub-48', 'sub-49']

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

from cross_subject_encoding.data import things_eeg1 as e1
from cross_subject_encoding.utils.eeg1_pilot import sha256_file, source_file
assert sha256_file(C.find_artifact("eeg1_paper_kpca1000.npz")) == "61d36682db158fc622efe721c7759f70f52868a828aa9a66d395fdff7318ec1f"
with np.load(C.find_artifact("eeg1_paper_kpca1000.npz")) as f:
    z_train, z_test = f["train"].astype(np.float64), f["test"].astype(np.float64)
subsets = json.loads(C.find_artifact("eeg1_k_subsets.json").read_text())
people = sorted(PEOPLE)  # the 22 sources only; the 24 confirmation targets are never opened
y_train, bio_test = {}, {}
for s in people:
    for part in ("training", "test"):
        rel = te.load_part(source_file(Path("/kaggle/input"), s, part, people))
        if part == "training":
            y_train[s], times = rel["data"][:, 0], rel["times"]
        else:
            bio_test[s] = rel["data"]
order = lambda s, d, k: np.asarray(subsets["orders"][s][str(d)])[:k]
print(len(people), 'people loaded |', f'{(time.monotonic() - T0) / 60:.1f} min')

In [ ]:
N_TRAIN, N_TEST = len(z_train), len(z_test)
n_ch, n_t = 17, 100
query = np.concatenate([z_train, z_test])
src_train = np.empty((len(people), N_TRAIN, n_ch * n_t), np.float32)
src_test = np.empty((len(people), N_TEST, n_ch * n_t), np.float32)
for i, s in enumerate(people):
    pred = pl.ols_paper(z_train, y_train[s].reshape(N_TRAIN, -1), query)
    src_train[i], src_test[i] = pred[:N_TRAIN], pred[N_TRAIN:]
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
match = (times >= FIXED["match_window_s"][0]) & (times < FIXED["match_window_s"][1])
win = ha.window_index(times, FIXED["map_window_ms"])
t3 = lambda a, n: np.asarray(a, np.float64).reshape(n, n_ch, n_t)

def shift(a, s):
    # move a response s samples later (s > 0: the person responds later than the crowd), padding with the edge value
    if s == 0:
        return a
    out = np.empty_like(a)
    if s > 0:
        out[..., s:], out[..., :s] = a[..., :-s], a[..., :1]
    else:
        out[..., :s], out[..., s:] = a[..., -s:], a[..., -1:]
    return out

def best_shift(a_k, y_k):
    # the lag whose shifted average correlates best with the person's labelled responses, 0-400 ms, across images
    def score(s):
        p = shift(a_k, s)[..., match]
        p, y = p - p.mean(0), y_k[..., match] - y_k[..., match].mean(0)
        return float(np.nanmean((p * y).sum(0) / np.sqrt((p ** 2).sum(0) * (y ** 2).sum(0))))
    return max(FIXED["shifts"], key=score)

rows, shift_rows = [], []
for i, person in enumerate(people):
    t = time.monotonic()
    others = [j for j in range(len(people)) if j != i]
    avg_train, avg_t3 = src_train[others].mean(0), t3(src_test[others].mean(0), N_TEST)
    preds = {"avg|all|all": avg_t3}
    for d in FIXED["draws"]:
        for k in FIXED["ks"]:
            idx = order(person, d, k)
            a_k, y_k = t3(avg_train[idx], k), np.asarray(y_train[person][idx], np.float64)
            s = best_shift(a_k, y_k)
            shift_rows.append({"person": person, "k": k, "draw": d, "shift_samples": s})
            preds[f"h4b|{k}|{d}"] = h4b.apply_map(h4b.fit_map(a_k, y_k, win, seed=d)[0], avg_t3, win)
            preds[f"avg_shift|{k}|{d}"] = shift(avg_t3, s)
            preds[f"h4b_shift|{k}|{d}"] = h4b.apply_map(h4b.fit_map(shift(a_k, s), y_k, win, seed=d)[0], shift(avg_t3, s), win)
    preds = {name: np.asarray(p).reshape(N_TEST, n_ch, n_t) for name, p in preds.items()}
    if not all(np.isfinite(p).all() for p in preds.values()):
        raise ValueError(f"{person}: nonfinite prediction")
    scored = paper_correlation(preds, bio_test[person], n_iter=100)
    for name, curve in scored["correlation"].items():
        method, k, d = name.split("|")
        row = {"person": person, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean())}
        row.update({f"r_w{w}": float(curve[:, win == w].mean()) for w in np.unique(win)})
        rows.append(row)
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / f"{TAG}_scores.csv", index=False)
    pd.DataFrame(shift_rows).to_csv(C.METRICS_DIR / f"{TAG}_shifts.csv", index=False)
    print(person, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
frame, shifts = pd.DataFrame(rows), pd.DataFrame(shift_rows)
def per_person(method, k):
    f = frame[(frame.method == method) & (frame.k == str(k))]
    return f.groupby("person")["r_100_200ms"].mean().reindex(people).to_numpy()
def ci(x):
    se = x.std(ddof=1) / np.sqrt(len(x))
    t = stats.t.ppf(0.975, len(x) - 1)
    return {"mean": float(x.mean()), "lower95": float(x.mean() - t * se), "upper95": float(x.mean() + t * se),
            "positive": int((x > 0).sum()), "n": int(len(x))}
avg = per_person("avg", "all")
mean_k = lambda method: np.mean([per_person(method, k) for k in FIXED["ks"]], 0)
h, a_s, h_s = mean_k("h4b"), mean_k("avg_shift"), mean_k("h4b_shift")
summary = {"dataset": DATASET, "post_hoc": "development",
           "shift_alone_minus_avg": ci(a_s - avg), "map_minus_avg": ci(h - avg), "map_after_shift": ci(h_s - a_s),
           "shift_plus_map_minus_map": ci(h_s - h), "spatial_share": float((h_s - a_s).mean() / (h - avg).mean()),
           "by_k": {k: {"shift_alone_minus_avg": ci(per_person("avg_shift", k) - avg),
                        "shift_plus_map_minus_map": ci(per_person("h4b_shift", k) - per_person("h4b", k))} for k in FIXED["ks"]}}
ms = 1000 * (times[1] - times[0])
per = shifts.groupby("person")["shift_samples"]
summary["shift_ms"] = {"mean": float(shifts.shift_samples.mean() * ms), "share_zero": float((shifts.shift_samples == 0).mean()),
                       "by_value": {int(v * ms): int(c) for v, c in shifts.shift_samples.value_counts().sort_index().items()},
                       "within_person_sd": float(per.std(ddof=1).mean() * ms), "between_person_sd": float(per.mean().std(ddof=1) * ms)}
summary["rule"] = ("descriptive: the personal part is spatial if the map's gain after the shift keeps at least 80% of "
                   "its gain without it and its 95% lower bound is above 0; latency adds to the map if shift + map minus map "
                   "has a 95% lower bound above 0")
summary["reading"] = {"spatial": bool(summary["spatial_share"] >= 0.8 and summary["map_after_shift"]["lower95"] > 0),
                      "latency_adds": bool(summary["shift_plus_map_minus_map"]["lower95"] > 0)}
(C.METRICS_DIR / f"{TAG}_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
labels = ["shift alone\n− average", "map\n− average", "map after shift\n− shifted average", "shift + map\n− map"]
vals = [a_s - avg, h - avg, h_s - a_s, h_s - h]
for j, v in enumerate(vals):
    ax[0].scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.15, 0.15, len(v)), v, s=10, alpha=0.6)
    c = ci(v)
    ax[0].errorbar(j, c["mean"], yerr=[[c["mean"] - c["lower95"]], [c["upper95"] - c["mean"]]], fmt="o", color="black")
ax[0].axhline(0, color="black", lw=0.6)
ax[0].set(xticks=range(4), title=f"{DATASET}: latency or channel mixing? (k = 100-500)", ylabel="difference in r, 100-200 ms")
ax[0].set_xticklabels(labels, fontsize=7)
vc = shifts.shift_samples.value_counts().sort_index()
ax[1].bar(vc.index * ms, vc.values, width=0.8 * ms)
ax[1].set(xlabel="chosen shift (ms; positive = later than the crowd)", ylabel="count (person × k × draw)", title="chosen lags")
fig.tight_layout()
fig.savefig(C.METRICS_DIR / f"{TAG}_figure.png", dpi=150)
print(json.dumps({k: summary[k] for k in ("shift_alone_minus_avg", "map_minus_avg", "map_after_shift", "shift_plus_map_minus_map", "spatial_share", "shift_ms", "reading")}, indent=1))

In [ ]:
export_to_dataset(slug="cross-subject-brain-encoding-17a-outputs", title="XSUB 17a latency outputs",
                  files=[C.METRICS_DIR / f"{TAG}_{x}" for x in ("summary.json", "scores.csv", "shifts.csv", "figure.png")],
                  include_dirs=[], message="latency or channel mixing")